In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from numpy import sqrt

In [ ]:
u_data = pd.read_csv("/content/U_DATA.csv")
u_data.head(5)

,ID пользователей,ID фильма,Оценка,Дата,Временная метка
0,196,242,3,04.12.1997 15:55:49,881250949
1,186,302,3,04.04.1998 19:22:22,891717742
2,22,377,1,07.11.1997 7:18:36,878887116
3,244,51,2,27.11.1997 5:02:03,880606923
4,166,346,1,02.02.1998 5:33:16,886397596


In [ ]:
#Из этого кода мы видим, что наибольшее количество оценок поставил пользователь с ID №405.
count_values = u_data['ID пользователей'].value_counts()
count_values.head(5)

,count
ID пользователей,
405,737
655,685
13,636
450,540
276,518


In [ ]:
# Оставляем только те фильмы (их ID), которые оценил юзер 405. Здесь мы можем себя проверить. И в первой выборке и во второй у нас 737 значений.
user = u_data[u_data['ID пользователей'] == 405]
user.head()

,ID пользователей,ID фильма,Оценка,Дата,Временная метка
12276,405,56,4,23.01.1998 8:41:51,885544911
12383,405,592,1,23.01.1998 9:44:30,885548670
12430,405,1582,1,23.01.1998 9:44:30,885548670
12449,405,171,1,23.01.1998 9:59:04,885549544
12460,405,580,1,23.01.1998 9:24:07,885547447


In [ ]:
# Загрузим датафрейм с фильмами и жанрами и соеденим с таблицей по самому активному пользователю.
u_item = pd.read_csv("/content/U_item_Лучший - Фильмы (1).csv")
u_item.head(5)

,ID фильма,Название фильма,Дата выхода фильма,Месяц выхода фильма,Год выхода фильма,Неизвестен,Боевик,Приключения,Анимация,Детский,...,Фэнтези,Фильм-Нуар,Ужасы,Мюзикл,Мистика,Мелодрама,Научная фантастика,Триллер,Военный,Вестерн
0,1,Toy Story (1995),1.0,Jan,1995.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,GoldenEye (1995),1.0,Jan,1995.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,3,Four Rooms (1995),1.0,Jan,1995.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,4,Get Shorty (1995),1.0,Jan,1995.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,Copycat (1995),1.0,Jan,1995.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [ ]:
# Создадим датафрейм по самому активному пользователю и фильмам с жанрами.
join_df_user_405 = pd.merge(user, u_item, on='ID фильма', how='left')
join_df_user_405.head()

,ID пользователей,ID фильма,Оценка,Дата,Временная метка,Название фильма,Дата выхода фильма,Месяц выхода фильма,Год выхода фильма,Неизвестен,...,Фэнтези,Фильм-Нуар,Ужасы,Мюзикл,Мистика,Мелодрама,Научная фантастика,Триллер,Военный,Вестерн
0,405,56,4,23.01.1998 8:41:51,885544911,Pulp Fiction (1994),1.0,Jan,1994.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,405,592,1,23.01.1998 9:44:30,885548670,True Crime (1995),1.0,Jan,1995.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,405,1582,1,23.01.1998 9:44:30,885548670,T-Men (1947),1.0,Jan,1947.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,405,171,1,23.01.1998 9:59:04,885549544,Delicatessen (1991),1.0,Jan,1991.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,405,580,1,23.01.1998 9:24:07,885547447,Englishman Who Went Up a Hill,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Уберем пропуски.
#Посчитаем общее количество оценок и суммарную оценку по каждому фильму.
df_All = pd.merge(u_item, u_data, on='ID фильма', how='left')
df_All = (df_All[['ID пользователей', 'ID фильма', 'Оценка', 'Название фильма', 'Год выхода фильма', 'Неизвестен', 'Боевик', 'Приключения', 'Анимация', 'Детский', 'Комедийный', 'Криминальный', 'Документальный', 'Драма', 'Фэнтези', 'Фильм-Нуар', 'Ужасы', 'Мюзикл', 'Мистика', 'Мелодрама', 'Научная фантастика', 'Триллер', 'Военный', 'Вестерн']])

df_All.head(5)

,ID пользователей,ID фильма,Оценка,Название фильма,Год выхода фильма,Неизвестен,Боевик,Приключения,Анимация,Детский,...,Фэнтези,Фильм-Нуар,Ужасы,Мюзикл,Мистика,Мелодрама,Научная фантастика,Триллер,Военный,Вестерн
0,308,1,4,Toy Story (1995),1995.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,287,1,5,Toy Story (1995),1995.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,148,1,4,Toy Story (1995),1995.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,280,1,4,Toy Story (1995),1995.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,66,1,3,Toy Story (1995),1995.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df_All[['Год выхода фильма', 'Неизвестен', 'Боевик', 'Приключения', 'Анимация', 'Детский', 'Комедийный', 'Криминальный', 'Документальный', 'Драма', 'Фэнтези', 'Фильм-Нуар', 'Ужасы', 'Мюзикл', 'Мистика', 'Мелодрама', 'Научная фантастика', 'Триллер', 'Военный', 'Вестерн']] = df_All[['Год выхода фильма', 'Неизвестен', 'Боевик', 'Приключения', 'Анимация', 'Детский', 'Комедийный', 'Криминальный', 'Документальный', 'Драма', 'Фэнтези', 'Фильм-Нуар', 'Ужасы', 'Мюзикл', 'Мистика', 'Мелодрама', 'Научная фантастика', 'Триллер', 'Военный', 'Вестерн']].fillna(value=0)
df_All.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ID пользователей    100000 non-null  int64  
 1   ID фильма           100000 non-null  int64  
 2   Оценка              100000 non-null  int64  
 3   Название фильма     100000 non-null  object 
 4   Год выхода фильма   100000 non-null  float64
 5   Неизвестен          100000 non-null  float64
 6   Боевик              100000 non-null  float64
 7   Приключения         100000 non-null  float64
 8   Анимация            100000 non-null  float64
 9   Детский             100000 non-null  float64
 10  Комедийный          100000 non-null  float64
 11  Криминальный        100000 non-null  float64
 12  Документальный      100000 non-null  float64
 13  Драма               100000 non-null  float64
 14  Фэнтези             100000 non-null  float64
 15  Фильм-Нуар          100000 non-null

In [ ]:
df_All['Общее кол-во'] = df_All.groupby(['Оценка'])['Название фильма'].transform('count')
df_All['Суммарная оценка'] = df_All.groupby(['Название фильма'])['Оценка'].transform('sum')
df_All.head()

,ID пользователей,ID фильма,Оценка,Название фильма,Год выхода фильма,Неизвестен,Боевик,Приключения,Анимация,Детский,...,Ужасы,Мюзикл,Мистика,Мелодрама,Научная фантастика,Триллер,Военный,Вестерн,Общее кол-во,Суммарная оценка
0,308,1,4,Toy Story (1995),1995.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34174,1753
1,287,1,5,Toy Story (1995),1995.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21201,1753
2,148,1,4,Toy Story (1995),1995.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34174,1753
3,280,1,4,Toy Story (1995),1995.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34174,1753
4,66,1,3,Toy Story (1995),1995.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,27145,1753


In [ ]:
df_All['Общее кол-во'] = df_All['Общее кол-во'].astype(float)
df_All['Суммарная оценка'] = df_All['Суммарная оценка'].astype(float)

In [ ]:
# Сформируем X и y из переменной, выбрав нужные колонки:
df_All.columns

Index(['ID пользователей', 'ID фильма', 'Оценка', 'Название фильма',
       'Год выхода фильма', 'Неизвестен', 'Боевик', 'Приключения', 'Анимация',
       'Детский', 'Комедийный', 'Криминальный', 'Документальный', 'Драма',
       'Фэнтези', 'Фильм-Нуар', 'Ужасы', 'Мюзикл', 'Мистика', 'Мелодрама',
       'Научная фантастика', 'Триллер', 'Военный', 'Вестерн', 'Общее кол-во',
       'Суммарная оценка'],
      dtype='object')

In [ ]:
X, y = df_All[['Год выхода фильма', 'Неизвестен', 'Боевик', 'Приключения', 'Анимация',
       'Детский', 'Комедийный', 'Криминальный', 'Документальный', 'Драма',
       'Фэнтези', 'Фильм-Нуар', 'Ужасы', 'Мюзикл', 'Мистика', 'Мелодрама',
       'Научная фантастика', 'Триллер', 'Военный', 'Вестерн', 'Общее кол-во',
       'Суммарная оценка']], df_All['ID фильма']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_test.head()

,Год выхода фильма,Неизвестен,Боевик,Приключения,Анимация,Детский,Комедийный,Криминальный,Документальный,Драма,...,Ужасы,Мюзикл,Мистика,Мелодрама,Научная фантастика,Триллер,Военный,Вестерн,Общее кол-во,Суммарная оценка
56740,1996.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,34174.0,1140.0
21827,1979.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,27145.0,652.0
49938,1996.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34174.0,529.0
2953,1996.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34174.0,1107.0
89472,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34174.0,364.0


In [ ]:
# Создадим и обучим модель линейной регрессии.
model = LinearRegression()

In [ ]:
model.fit(X_train, y_train)

LinearRegression()

In [ ]:
# Оценим качество на выборке для обучения
from sklearn.metrics import mean_squared_error

In [ ]:
mean_squared_error(y_train, model.predict(X_train))

73242.29751466021

In [ ]:
# Оценим качество на тестовой выборке
mean_squared_error(y_test, model.predict(X_test))

73242.15691024267